
# Enrich nbastatsv3 Shot Data (Full) — Improved Shot Clock + Hybrid Contest Classifier

This notebook loads your **`nbastatsv3_2024.csv`** play-by-play, then:
- Reconstructs an **approximate shot clock** per shot (`SHOT_CLOCK_APPROX`) using an improved possession reset model.
- Classifies field-goal shots into **likely_contested / borderline / likely_open** with a **hybrid** heuristic that uses
  **description + actionType + subType + distance + x/y** (NBA tenths-of-feet for corner-3) + late-clock.

It also includes helpers to:
- Enrich **one player in one game**
- Enrich **one player across all their games**
- Optionally export a **full-season** enriched shots CSV in batches


## ⏱️ Approximate Shot Clock (`SHOT_CLOCK_APPROX`)

### What it is

The **shot clock** is the 24-second timer (14 seconds on offensive rebounds) that governs how long a team has to attempt a shot. The official NBA public dataset (`nbastatsv3_2024.csv`) doesn't include the actual shot clock value at the time of a shot.

So we **approximate** it by reconstructing possessions from play-by-play events.

### How it works (v4)

1. **Track possession by team**: Each reset event is tagged with the team that gains possession, ensuring we only use resets relevant to the shooting team.

2. **Resolve missing team IDs**: If `teamId` is missing/0, infer it from `teamTricode` or team names in the description (e.g., "Bulls Rebound").

3. **Track last shooting team**: To determine if a rebound is offensive or defensive, we track which team took the last shot or free throw.

4. **Reset events (new shot clock):**

   | Event | Possession Goes To | Shot Clock |
   |-------|-------------------|------------|
   | **Made field goal** | Opposing team | 24s |
   | **Turnover** (incl. offensive goaltending) | Opposing team | 24s |
   | **Jump ball** | Winning team | 24s |
   | **Defensive rebound** | Rebounding team | 24s |
   | **Offensive rebound** | Rebounding team | 14s |
   | **Defensive violations** (kicked ball, def 3-sec, goaltending) | Offense keeps ball | 14s |
   | **Offensive foul / charge / illegal screen** | Opposing team | 24s |
   | **Defensive foul (clock <= 14)** | Offense keeps ball | 14s |
   | **Defensive foul (clock > 14)** | Offense keeps ball | keep current (`keep_clock`) |
   | **Final free throws** (1/1, 2/2, 3/3, technical, flagrant) | Opposing team | 24s |
   | **Period start** | First team with possession | 24s |

5. **Offensive vs Defensive Rebound Detection**:
   - **Offensive rebound** = rebounding team is the SAME team that took the shot -> 14s reset
   - **Defensive rebound** = rebounding team is DIFFERENT from shooting team -> 24s reset
   - **Important**: The `(Off:X Def:Y)` in rebound descriptions are CUMULATIVE player stats for the game, NOT indicators of whether the specific rebound is offensive or defensive!

6. **Ordering matters**: Resets are ordered by `ABS_TIME`, then `actionNumber` to handle play-by-play ordering quirks.

7. **Shot clock calculation:**

   * For each shot, find the **last reset event that gave possession to the shooting team**
   * Calculate elapsed time: `delta = shot_time - reset_time`
   * Remaining shot clock: `SHOT_CLOCK_APPROX = cap - delta`
   * If `delta > cap`, we record `SHOT_CLOCK_APPROX = NaN` and mark `SHOT_CLOCK_SOURCE = stale_reset(cap)`
   * Example: If last reset was at 8:00 in the quarter (24s reset), and the shot was at 7:50, then `delta = 10`, and `SHOT_CLOCK_APPROX = 24 - 10 = 14`.

8. **Fallback for period starts:**

   * If no reset is found for a team early in a period, we assume 24 seconds from period start.

### Bug fixes / refinements

| Bug | Problem | Fix |
|-----|---------|-----|
| **#1** | Made shots used themselves as reset events, giving 0s shot clock | Use strict `< t` comparison and check `actionNumber` |
| **#2** | Formula returned elapsed time instead of remaining time | Changed to `cap - delta` |
| **#3** | Didn't track which team had possession | Added `POSS_TEAM` column - only uses resets that gave possession to the shooting team |
| **#4** | `"off:"` in desc matched ALL rebounds because `(Off:0 Def:1)` contains `off:` | Removed this check entirely |
| **#5** | N/A (merged into #6) | - |
| **#6** | `Off:X` in descriptions is cumulative player stats, not this rebound's type | Track last shooting team and compare to rebounding team |
| **#7** | Out-of-order PBP events caused stale resets | Sort resets by `ABS_TIME` then `actionNumber` |
| **#8** | Missing/0 team IDs for team rebounds/turnovers | Infer team from tricode or team name in description |
| **#9** | Defensive fouls always forced 14s | Only reset to 14 when remaining <= 14; otherwise keep current (`keep_clock`) |

### Limitations

* Doesn't track inbound delays or retained possession after technical/flagrant FTs.
* Timeouts pause the clock but don't change possession - we don't explicitly model this.
* PBP clock is rounded to 0.1s and can drift; some true possessions will still be tagged as `stale_reset(...)` when the reset event is missing or ambiguous.

---

## 🏀 Contested/Open Heuristic (`contest_score`, `contest_label`, `contest_reasons`)

Since defender distance (`CLOSE_DEF_DIST`) isn't in this dataset, we use **play-by-play text, action type, sub type, shot attributes, and coordinates** to *estimate* whether a shot was contested.

### How it works

1. **Action Type / SubType / Description**

   * **Contested-like** (+2): Pull-Up, Step Back, Fadeaway, Driving Layup, Driving Floating, Hook, Dunk, Putback, Alley-Oop, Reverse, Turnaround, etc.
   * **Open-like** (–1): Catch-and-Shoot, Spot Up, Generic Jump Shot, Regular.
   * **Description text** is also parsed to catch "step back", "fadeaway", "catch-and-shoot" even when subType is `Unknown`.

2. **Shot Location / Distance**

   * **At-rim / very close (≤5 ft)** (+1) → usually more defended.
   * **Mid-range (8–16 ft)** (+1) → often tightly contested.
   * **Deep above-the-break 3s (≥27 ft)** (–1) → more likely to be open.

3. **Corner vs Above-the-Break** (using NBA x/y coordinates, tenths of feet)

   * Corner-3 defined as **`|x| ≥ 220` and `y ≤ 50`**.
   * Corner catch-and-shoot or spot-up looks are nudged more open (–1).

4. **Shot Clock Context**

   * If `SHOT_CLOCK_APPROX ≤ 5` (+1): End-of-clock situations usually force **heavily contested or rushed** shots.

5. **Scoring System (`contest_score`):**

   * Start at 0
   * Add/subtract points from rules above
   * Examples:

     * Driving layup with 2 seconds left → +2 (driving) +1 (close range) +1 (late clock) = **+4**
     * Catch-and-shoot 28′ above-the-break three with 14 seconds left → –1 (catch-and-shoot) –1 (deep 3) = **–2**

6. **Labels (`contest_label`):**

   * `≥ 2` → **likely\_contested**
   * `= 1` → **borderline**
   * `≤ 0` → **likely\_open**

7. **Reasons (`contest_reasons`):**

   * Text log of why the shot got its score.
   * Example: `"subType: contested-like, distance: at-rim/very close, clock: late (<=5s)"`

---

## 🔎 Example Output

| Description                  | Shot Clock Approx | Contest Score | Contest Label     | Contest Reasons                                              |
| ---------------------------- | ----------------- | ------------- | ----------------- | ------------------------------------------------------------ |
| MISS 29′ Pullup 3            | 12                | +1            | borderline        | subType: contested-like, text: catch/spot                    |
| Driving Reverse Layup (made) | 18                | +3            | likely\_contested | subType: contested-like, distance: at-rim/very close         |
| Catch-and-shoot 27′ 3 (made) | 15                | –2            | likely\_open      | subType: open-like, distance: deep above-break 3, coords: corner-3 |

---

✅ **Summary:**

* **Approximate Shot Clock** = estimate of time **remaining** on the shot clock when the shot was taken, reconstructed from possession resets with proper team tracking.
* **Contested/Open Heuristic** = hybrid rule-based system that classifies shots into **likely\_open**, **borderline**, or **likely\_contested**, using **action type, subType, description text, shot distance, coordinates, and shot clock pressure**.


## 🏀 Court Coordinates & Zones (xLegacy / yLegacy)

### Our play-by-play uses **NBA shot coordinates in tenths of feet**:

- **xLegacy**: –250 to +250 (sideline ↔ sideline)  
- **yLegacy**: 0 to 470 (baseline → half-court)  
- **Basket**: (0, 0)

**Corner-3 rule used in the heuristic**: `|xLegacy| ≥ 220` **and** `yLegacy ≤ 50`  
All other 3PT shots are treated as **above-the-break**.

### Zones referenced by the heuristic

- **Paint (Key)**: roughly inside the lane (~0–15 ft) — typically tighter defense  
- **Mid-range**: ~8–16 ft annulus — often contested  
- **Corner-3 zone**: `|x|≥220 & y≤50` — catch-and-shoot here is often more open  
- **Above-the-break 3**: all other 3PTs; very deep (≥27 ft) are treated as more open


## 1. Setup

In [ ]:

# If needed:
# !pip install pandas numpy matplotlib

import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Load Data

In [ ]:

# CSV_PATH = "raw_data/nbastatsv3_2024.csv"        # <- update if needed
CSV_PATH = "raw_data/nbastatsv3_2024.csv"        # <- update if needed

all_data = pd.read_csv(CSV_PATH)
print("Loaded rows:", len(all_data))
display(all_data.head())


## 3. Shot-Clock Reconstruction (v3)

In [ ]:
REG_PERIOD_LEN = 12 * 60
OT_PERIOD_LEN  = 5 * 60

def _period_len_s(period:int)->int:
    return REG_PERIOD_LEN if period <= 4 else OT_PERIOD_LEN

def _abs_time(period:int, clock_iso:str)->int:
    # clock like 'PT11M43.00S'
    m = re.match(r"PT(\d+)M(\d+)\.\d+S", str(clock_iso))
    if not m:
        return None
    rem = int(m.group(1))*60 + int(m.group(2))
    return sum(_period_len_s(p) for p in range(1, period)) + (_period_len_s(period) - rem)

# Free-throw "final-of-sequence" subtypes that typically end a trip
_FINAL_FT = {
    "free throw 1 of 1", "free throw 2 of 2", "free throw 3 of 3",
    "free throw technical", "free throw technical 2 of 2",
    "free throw clear path 2 of 2",
    "free throw flagrant 1 of 1", "free throw flagrant 2 of 2", "free throw flagrant 3 of 3",
}

# Turnover subtypes to treat as possession change (24 reset to opponent)
_TURNOVER_POSSESSION = {
    "bad pass","lost ball","out of bounds - bad pass turnover","out of bounds lost ball turnover",
    "offensive foul turnover","offensive charge","shot clock turnover",
    "traveling","backcourt turnover","step out of bounds turnover",
    "double dribble","palming turnover","8 second violation","5 second violation",
    "illegal assist turnover","excess timeout turnover","basket from below turnover",
    "too many players turnover","punched ball turnover","inbound turnover","illegal screen turnover","offensive goaltending",
}

# Defensive violations that keep offense but reset to 14 (approx)
_DEFENSE_KEEPS_14 = {
    "kicked ball","kicked ball violation","defense 3 second","defensive goaltending"
}

def compute_shot_clock_v4(df: pd.DataFrame, game_id: int, player_id: int) -> pd.DataFrame:
    """
    Shot clock reconstruction v4 with proper offensive/defensive rebound detection.

    Key fix: Determine if a rebound is offensive or defensive by comparing the
    rebounding team to the team that took the last shot/free throw.
    - Offensive rebound = same team that shot -> 14 second reset
    - Defensive rebound = other team -> 24 second reset

    The (Off:X Def:Y) in descriptions are CUMULATIVE player stats for the game,
    NOT indicators of whether this specific rebound is offensive or defensive.
    """
    g = df[df["gameId"]==game_id].copy()
    g = g.sort_values(["period","actionNumber"]).reset_index(drop=True)

    # Absolute times
    g["ABS_TIME"] = g.apply(lambda r: _abs_time(int(r["period"]), r["clock"]), axis=1)

    # Get unique teams in game (exclude 0 which is used for period events)
    teams_in_game = [t for t in g[g["teamId"].notna()]["teamId"].unique() if t != 0]

    tricode_to_id = {}
    if "teamTricode" in g.columns:
        pairs = g[["teamId", "teamTricode"]].dropna()
        for tid, tri in pairs.values.tolist():
            if tid != 0 and tri:
                tricode_to_id[str(tri).upper()] = tid

    TEAM_NAME_TO_TRICODE = {
        "ATLANTA HAWKS": "ATL", "HAWKS": "ATL",
        "BOSTON CELTICS": "BOS", "CELTICS": "BOS",
        "BROOKLYN NETS": "BKN", "NETS": "BKN",
        "CHARLOTTE HORNETS": "CHA", "HORNETS": "CHA",
        "CHICAGO BULLS": "CHI", "BULLS": "CHI",
        "CLEVELAND CAVALIERS": "CLE", "CAVALIERS": "CLE", "CAVS": "CLE",
        "DALLAS MAVERICKS": "DAL", "MAVERICKS": "DAL", "MAVS": "DAL",
        "DENVER NUGGETS": "DEN", "NUGGETS": "DEN",
        "DETROIT PISTONS": "DET", "PISTONS": "DET",
        "GOLDEN STATE WARRIORS": "GSW", "WARRIORS": "GSW",
        "HOUSTON ROCKETS": "HOU", "ROCKETS": "HOU",
        "INDIANA PACERS": "IND", "PACERS": "IND",
        "LA CLIPPERS": "LAC", "LOS ANGELES CLIPPERS": "LAC", "CLIPPERS": "LAC",
        "LA LAKERS": "LAL", "LOS ANGELES LAKERS": "LAL", "LAKERS": "LAL",
        "MEMPHIS GRIZZLIES": "MEM", "GRIZZLIES": "MEM",
        "MIAMI HEAT": "MIA", "HEAT": "MIA",
        "MILWAUKEE BUCKS": "MIL", "BUCKS": "MIL",
        "MINNESOTA TIMBERWOLVES": "MIN", "TIMBERWOLVES": "MIN", "WOLVES": "MIN",
        "NEW ORLEANS PELICANS": "NOP", "PELICANS": "NOP",
        "NEW YORK KNICKS": "NYK", "KNICKS": "NYK",
        "OKLAHOMA CITY THUNDER": "OKC", "THUNDER": "OKC",
        "ORLANDO MAGIC": "ORL", "MAGIC": "ORL",
        "PHILADELPHIA 76ERS": "PHI", "76ERS": "PHI", "SIXERS": "PHI",
        "PHOENIX SUNS": "PHX", "SUNS": "PHX",
        "PORTLAND TRAIL BLAZERS": "POR", "TRAIL BLAZERS": "POR", "BLAZERS": "POR",
        "SACRAMENTO KINGS": "SAC", "KINGS": "SAC",
        "SAN ANTONIO SPURS": "SAS", "SPURS": "SAS",
        "TORONTO RAPTORS": "TOR", "RAPTORS": "TOR",
        "UTAH JAZZ": "UTA", "JAZZ": "UTA",
        "WASHINGTON WIZARDS": "WAS", "WIZARDS": "WAS",
    }
    team_name_to_id = {
        name: tricode_to_id[tri]
        for name, tri in TEAM_NAME_TO_TRICODE.items()
        if tri in tricode_to_id
    }

    def get_other_team(team):
        for t_id in teams_in_game:
            if t_id != team:
                return t_id
        return None

    def infer_team_from_desc(desc: str):
        if not desc:
            return None
        desc_u = str(desc).upper()
        for tri, tid in tricode_to_id.items():
            if tri and re.search(r"\b" + re.escape(tri) + r"\b", desc_u):
                return tid
        for name, tid in team_name_to_id.items():
            if name and re.search(r"\b" + re.escape(name) + r"\b", desc_u):
                return tid
        return None

    def infer_team_from_event(team, team_tricode, desc):
        if pd.notna(team) and team != 0:
            return team
        if pd.notna(team_tricode):
            tri = str(team_tricode).upper()
            if tri in tricode_to_id:
                return tricode_to_id[tri]
        return infer_team_from_desc(desc)

    def is_team_rebound_desc(desc: str):
        if not desc:
            return False
        desc_l = str(desc).lower()
        if "rebound" not in desc_l:
            return False
        if "team rebound" in desc_l:
            return True
        desc_u = desc_l.upper()
        for tri in tricode_to_id.keys():
            if tri and re.search(r"\b" + re.escape(tri) + r"\b", desc_u):
                return True
        return False

    # Track resets with team possession info
    # Format: (period, abs_time, reset_val, note, action_number, team_with_possession)
    resets = []

    # Track the last team that shot (for rebound classification)
    last_shot_team = {p: None for p in range(1, 10)}  # per period

    # Track last reset by team to estimate remaining clock at fouls
    last_reset_by_team = {}

    def add_reset(per, t, cap, note, action_num, team_id):
        if team_id is None or pd.isna(team_id) or team_id == 0:
            return
        resets.append((per, t, cap, note, action_num, team_id))
        last_reset_by_team[(per, team_id)] = (t, cap)

    off_foul_keywords = [
        "offensive foul","offensive","charge","charging","illegal screen",
        "moving screen","player control","offensive charge"
    ]
    def_foul_keywords = [
        "loose ball foul","away from play","clear path","take foul",
        "shooting foul","personal foul","blocking foul","reach-in foul","holding foul"
    ]

    for _, ev in g.iterrows():
        per   = int(ev["period"])
        t     = ev["ABS_TIME"]
        aType = str(ev["actionType"]).strip().lower()
        sType = str(ev["subType"]).strip().lower() if pd.notna(ev.get("subType")) else ""
        team  = ev.get("teamId", np.nan)
        team_tri = ev.get("teamTricode", np.nan)
        action_num = ev.get("actionNumber", np.nan)
        desc = str(ev.get("description",""))
        desc_l = desc.lower()

        # If team is missing/0, try to infer from tricode or description
        if pd.isna(team) or team == 0:
            if aType not in ["period"]:
                inferred = infer_team_from_event(team, team_tri, desc)
                if inferred is not None:
                    team = inferred

        # Period start: skip, possession determined by jump ball
        if aType == "period" and sType == "start":
            last_shot_team[per] = None
            continue

        # Jump ball (possession to winner -> 24)
        if aType == "jump ball":
            if pd.notna(team) and team != 0:
                add_reset(per, t, 24, "jumpball", action_num, team)
            continue

        # Timeout: doesn't change possession
        if aType == "timeout":
            continue

        # Track shots and free throws (for rebound classification)
        if aType in ["made shot", "missed shot"]:
            if pd.notna(team) and team != 0:
                last_shot_team[per] = team
        if aType == "free throw":
            if pd.notna(team) and team != 0:
                last_shot_team[per] = team

        # Treat team rebound descriptions as rebounds even if actionType isn't rebound
        is_team_reb = (aType != "rebound") and is_team_rebound_desc(desc)

        # Rebounds - determine offensive vs defensive by comparing to last shot team
        if aType == "rebound" or is_team_reb:
            reb_team = team
            shooter = last_shot_team.get(per)
            if pd.isna(reb_team) or reb_team == 0:
                # If team missing, infer from last shot
                if shooter is not None:
                    if "offensive" in sType:
                        reb_team = shooter
                    elif "defensive" in sType:
                        reb_team = get_other_team(shooter)
                    else:
                        reb_team = get_other_team(shooter)

            # Check subType first (most reliable when available)
            if "offensive" in sType:
                is_off = True
            elif "defensive" in sType:
                is_off = False
            elif pd.notna(reb_team) and pd.notna(shooter):
                # Offensive rebound = same team that shot
                is_off = (reb_team == shooter)
            else:
                # Default to defensive if can't determine
                is_off = False

            if pd.notna(reb_team) and reb_team != 0:
                if is_off:
                    # Offensive rebound: same team keeps ball, reset to 14
                    add_reset(per, t, 14, "off_reb", action_num, reb_team)
                else:
                    # Defensive rebound: rebounding team gets ball with 24s
                    add_reset(per, t, 24, "def_reb", action_num, reb_team)
            continue

        # Made shots: OTHER team gets possession with 24s
        if aType == "made shot":
            shot_team = infer_team_from_event(team, team_tri, desc)
            other_team = get_other_team(shot_team) if shot_team is not None else None
            if other_team is not None:
                add_reset(per, t, 24, "made_fg", action_num, other_team)
            continue

        # Turnovers: OTHER team gets possession with 24s
        if aType == "turnover":
            off_team = infer_team_from_event(team, team_tri, desc) or last_shot_team.get(per)
            if sType in _TURNOVER_POSSESSION or sType == "" or sType == "regular":
                other_team = get_other_team(off_team) if off_team is not None else None
                if other_team is not None:
                    add_reset(per, t, 24, "turnover", action_num, other_team)
            continue

        # Defensive violations -> offense keeps ball, reset to 14
        if aType == "violation" and sType in _DEFENSE_KEEPS_14:
            poss_team = None
            if pd.notna(team) and team != 0:
                poss_team = get_other_team(team)
            else:
                poss_team = last_shot_team.get(per)
            if poss_team is not None:
                add_reset(per, t, 14, "def_violation_14", action_num, poss_team)
            continue

        # Fouls - approximate resets (defensive foul -> 14, offensive foul -> 24 to other team)
        if aType == "foul":
            sdesc = (sType + " " + desc_l).strip()
            if any(k in sdesc for k in off_foul_keywords):
                offender = infer_team_from_event(team, team_tri, desc) or last_shot_team.get(per)
                other_team = get_other_team(offender) if offender is not None else None
                if other_team is not None:
                    add_reset(per, t, 24, "off_foul", action_num, other_team)
            else:
                poss_team = None
                if pd.notna(team) and team != 0:
                    poss_team = get_other_team(team)
                else:
                    poss_team = last_shot_team.get(per)
                if poss_team is not None:
                    rem = None
                    last_cap = None
                    last = last_reset_by_team.get((per, poss_team))
                    if last is not None and pd.notna(t):
                        last_t, last_cap = last
                        if pd.notna(last_t):
                            rem = last_cap - (t - last_t)
                            if rem is not None and (rem > last_cap or rem <= 0):
                                rem = None
                    if rem is None or rem <= 14:
                        add_reset(per, t, 14, "def_foul_14", action_num, poss_team)
                    else:
                        # Keep existing clock; record remaining time (bounded to last cap)
                        cap = int(min(last_cap, rem)) if last_cap is not None else int(rem)
                        add_reset(per, t, cap, "keep_clock", action_num, poss_team)
            continue

        # Free throws (end-of-trip): OTHER team gets possession with 24s
        if aType == "free throw":
            if sType in _FINAL_FT:
                ft_team = infer_team_from_event(team, team_tri, desc) or last_shot_team.get(per)
                other_team = get_other_team(ft_team) if ft_team is not None else None
                if other_team is not None:
                    add_reset(per, t, 24, "final_ft", action_num, other_team)
            continue

    resets_df = pd.DataFrame(resets, columns=["PERIOD","ABS_TIME","RESET_VAL","RESET_NOTE","ACTION_NUMBER","POSS_TEAM"])
    resets_df = resets_df.sort_values(["PERIOD","ABS_TIME","ACTION_NUMBER"]).reset_index(drop=True)

    # Player FG shots only
    shots = g[(g["personId"]==player_id) & (g["actionType"].str.contains("Shot", case=False, na=False))].copy()

    # Attach approx shot clock
    sc_vals, sc_notes = [], []
    for _, sh in shots.iterrows():
        per = int(sh["period"])
        t = sh["ABS_TIME"]
        shot_action_num = sh.get("actionNumber", np.nan)
        shot_team = infer_team_from_event(sh.get("teamId", np.nan), sh.get("teamTricode", np.nan), sh.get("description", ""))

        # Find resets that:
        # 1. Are in the same period
        # 2. Happened BEFORE this shot (strict < or same time but lower action number)
        # 3. Gave possession to the shooting team
        prior = resets_df[
            (resets_df["PERIOD"] == per) &
            (resets_df["POSS_TEAM"] == shot_team) &
            (
                (resets_df["ABS_TIME"] < t) |
                ((resets_df["ABS_TIME"] == t) & (resets_df["ACTION_NUMBER"] < shot_action_num))
            )
        ]
        prior = prior.sort_values(["ABS_TIME","ACTION_NUMBER"])

        if prior.empty:
            # No reset found for this team - might be start of period
            # Default to 24 seconds from period start
            period_start_time = sum(_period_len_s(p) for p in range(1, per))
            delta = t - period_start_time
            if delta <= 24:
                sc_vals.append(max(0, 24 - delta))
                sc_notes.append("period_start(24)")
            else:
                sc_vals.append(np.nan)
                sc_notes.append("no_reset_found")
        else:
            last = prior.iloc[-1]
            delta = max(0, t - int(last["ABS_TIME"]))
            cap = int(last["RESET_VAL"])
            if delta > cap:
                sc_vals.append(np.nan)
                sc_notes.append(f"stale_reset({cap})")
            else:
                sc_vals.append(max(0, cap - delta))
                sc_notes.append(f"{last['RESET_NOTE']}({cap})")

    shots["SHOT_CLOCK_APPROX"] = sc_vals
    shots["SHOT_CLOCK_SOURCE"] = sc_notes
    return shots
# Alias for backward compatibility
compute_shot_clock_v3 = compute_shot_clock_v4










## 4. Helpers — Parse Shot Attributes from description/actionType/subType

In [ ]:
_SHOT_3PT_PAT = re.compile(
    r"\b3\s*pt\b|\b3pt\b|\b3-?point\b|\b3 pointer\b|\bthree point\b",
    re.IGNORECASE
)
_SHOT_2PT_PAT = re.compile(
    r"\b2\s*pt\b|\b2pt\b|\b2-?point\b|\b2 pointer\b",
    re.IGNORECASE
)
_FT_PAT = re.compile(r"\bfree throw\b", re.IGNORECASE)

def infer_shot_value_from_text(description:str, action_type:str, sub_type:str):
    """Return (shot_value, is_field_goal, is_free_throw) using text only."""
    d = (description or "")
    a = (action_type or "")
    s = (sub_type or "")
    text = f"{d} {a} {s}".lower()

    if _FT_PAT.search(text):
        return (1, False, True)

    if _SHOT_3PT_PAT.search(text):
        return (3, True, False)

    if _SHOT_2PT_PAT.search(text):
        return (2, True, False)

    # Generic FG cues default to 2
    fg_cues = (
        "made shot","missed shot","layup","dunk",
        "jumper","jump shot","hook","putback","tip","bank","fadeaway"
    )
    if any(cue in text for cue in fg_cues):
        return (2, True, False)

    return (np.nan, False, False)


## 5. Contest Classifier v3 — description + actionType + subType + coords + distance

In [ ]:

def classify_contest_level_v3(df: pd.DataFrame) -> pd.DataFrame:
    """
    Contest labeling that considers:
      - description, actionType, subType (to infer semantics and 2PT/3PT/FT)
      - shotDistance (or SHOT_DISTANCE) if present
      - xLegacy/yLegacy for corner-3 vs above-the-break (tenths of feet; abs(x)>=220 & y<=50 -> corner)
      - SHOT_CLOCK_APPROX for late-clock pressure
    """
    def nstr(x): return (str(x).strip().lower()) if pd.notna(x) else ""

    contested_like = {
        "pullup jump shot","running pull-up jump shot",
        "step back jump shot","turnaround fadeaway shot","fadeaway jump shot",
        "driving floating jump shot","floating jump shot","driving floating bank jump shot",
        "driving reverse layup shot","reverse layup shot",
        "driving layup shot","running layup shot","tip layup shot","putback layup shot",
        "driving dunk shot","running dunk shot","tip dunk shot","putback dunk shot","alley oop dunk shot","alley oop layup shot",
        "turnaround hook shot","driving hook shot","hook shot","hook bank shot",
        "turnaround bank hook shot","turnaround fadeaway bank jump shot","fadeaway bank shot","turnaround bank shot","jump bank shot"
    }
    open_like = {"jump shot","running jump shot","spot up","catch and shoot","catch-and-shoot","regular"}

    xcol, ycol = "xLegacy", "yLegacy"
    has_coords = (xcol in df.columns) and (ycol in df.columns)

    out_rows = []
    for _, r in df.iterrows():
        desc = nstr(r.get("description"))
        a    = nstr(r.get("actionType"))
        s    = nstr(r.get("subType"))

        shotv = r.get("shotValue", np.nan)
        if pd.isna(shotv):
            shotv, is_fg, is_ft = infer_shot_value_from_text(r.get("description",""), r.get("actionType",""), r.get("subType",""))
        else:
            is_fg, is_ft = True, False


        # Free throws: not labeled for contest
        if is_ft:
            new = r.copy()
            new["contest_score"] = np.nan
            new["contest_label"] = "n/a_free_throw"
            new["contest_reasons"] = "free throw"
            out_rows.append(new)
            continue

        if not is_fg:
            new = r.copy()
            new["contest_score"] = np.nan
            new["contest_label"] = "n/a_non_fg"
            new["contest_reasons"] = "not a field goal"
            out_rows.append(new)
            continue

        # Distance
        dist = r.get("shotDistance", r.get("SHOT_DISTANCE", np.nan))
        try: dist = float(dist) if pd.notna(dist) else np.nan
        except: dist = np.nan

        # Coords
        try:
            rx = float(r.get(xcol)) if has_coords else np.nan
            ry = float(r.get(ycol)) if has_coords else np.nan
        except:
            rx, ry = np.nan, np.nan

        is_3 = (shotv == 3)
        is_corner = bool(is_3 and pd.notna(rx) and pd.notna(ry) and (abs(rx) >= 220) and (ry <= 50))
        is_above_break = bool(is_3 and (not is_corner))

        score = 0
        why = []

        # 1) subType signals
        if s in contested_like:
            score += 2; why.append("subType: contested-like")
        if s in open_like:
            score -= 1; why.append("subType: open-like")

        # 2) text cues as backup for Unknown subType
        text = f"{desc} {a}".lower()
        if ("step back" in text) or ("pull-up" in text) or ("pullup" in text) or ("fadeaway" in text):
            score += 1; why.append("text: pressure-move")
        if ("catch and shoot" in text) or ("catch-and-shoot" in text) or ("spot up" in text) or ("spot-up" in text):
            score -= 1; why.append("text: catch/spot")

        # 3) distance nudges
        if pd.notna(dist):
            if dist <= 5:
                score += 1; why.append("distance: at-rim/very close")
            if 8 <= dist <= 16:
                score += 1; why.append("distance: mid-range")
            if is_above_break and dist >= 27:
                score -= 1; why.append("distance: deep above-break 3 (>=27ft)")

        # 4) corner nuance
        if is_corner:
            why.append("coords: corner-3")
            if ("catch and shoot" in text) or ("catch-and-shoot" in text) or ("spot up" in text) or ("spot-up" in text) or (s in {"jump shot","spot up","catch and shoot","catch-and-shoot"}):
                score -= 1; why.append("corner: catch/spot")

        # 5) late-clock
        sc = r.get("SHOT_CLOCK_APPROX", np.nan)
        try: sc = float(sc) if pd.notna(sc) else np.nan
        except: sc = np.nan
        if pd.notna(sc) and sc <= 5:
            score += 1; why.append("clock: late (<=5s)")

        # 6) label
        if score >= 2: label = "likely_contested"
        elif score == 1: label = "borderline"
        else: label = "likely_open"

        new = r.copy()
        new["shotValue"] = shotv
        new["contest_score"] = score
        new["contest_label"] = label
        new["contest_reasons"] = ", ".join(why)
        out_rows.append(new)

    return pd.DataFrame(out_rows).reset_index(drop=True)


## 6. Enrichment Helpers (one game, one player / all games)

In [ ]:

def enrich_player_game_v3(df: pd.DataFrame, game_id: int, player_id: int) -> pd.DataFrame:
    shots = compute_shot_clock_v3(df, game_id=game_id, player_id=player_id)
    return classify_contest_level_v3(shots)

def enrich_player_all_games_v3(df: pd.DataFrame, player_id: int, game_ids=None) -> pd.DataFrame:
    player_events = df[df["personId"]==player_id]
    all_gids = sorted(player_events["gameId"].unique())
    if game_ids is None:
        game_ids = all_gids
    else:
        game_ids = [gid for gid in game_ids if gid in all_gids]

    enriched = []
    for gid in game_ids:
        shots = enrich_player_game_v3(df, gid, player_id)
        if not shots.empty:
            enriched.append(shots)
    return pd.concat(enriched, ignore_index=True) if enriched else pd.DataFrame()


## 7. Example — All games for Steph Curry (201939)

In [ ]:

curry_all = enrich_player_all_games_v3(all_data, player_id=201939)
print("Total Curry shots:", len(curry_all))
display(curry_all.head(10))

# Simple plots (one figure each)
plt.figure()
curry_all["contest_label"].value_counts().plot(kind="bar", edgecolor="black")
plt.title("Curry shots by contest_label")
plt.xlabel("contest_label"); plt.ylabel("count")
plt.show()

# Save to CSV
output_file = "nbastatsv3_2024_steph_curry_shots.csv"
curry_all.to_csv(output_file, index=False)


## 8. Full-season export (all shots for all players, batched)

In [ ]:

# Chage this part to another output file if you need
# OUTPUT_CSV = "enriched_data/nbastatsv3_2024_enriched_shots.csv"
OUTPUT_CSV = "enriched_data/nbastatsv3_2024_enriched_shots.csv"

# Shot events only to reduce pairs
season_shots = all_data[all_data["actionType"].str.contains("Shot", case=False, na=False)].copy()

# Unique (gameId, personId) pairs
pairs = season_shots[["gameId","personId"]].drop_duplicates().values.tolist()

batch = []
written_any = False
for i, (gid, pid) in enumerate(pairs, 1):
    enriched = enrich_player_game_v3(all_data, game_id=gid, player_id=pid)
    if not enriched.empty:
        batch.append(enriched)
    # write every 50
    if i % 50 == 0:
        nonempty = [b for b in batch if (b is not None and not b.empty)]
        if nonempty:
            pd.concat(nonempty, ignore_index=True).to_csv(
                OUTPUT_CSV, mode=("w" if not written_any else "a"),
                header=(not written_any), index=False
            )
            written_any = True
        batch = []
        print(f"Processed {i}/{len(pairs)} pairs...")

# Flush remainder
nonempty = [b for b in batch if (b is not None and not b.empty)]
if nonempty:
    pd.concat(nonempty, ignore_index=True).to_csv(
        OUTPUT_CSV, mode=("w" if not written_any else "a"),
        header=(not written_any), index=False
    )
    written_any = True

print("Saved:", OUTPUT_CSV)


## 9. Sanity Check - GSW Total Points
Compare GSW field-goal points with https://www.basketball-reference.com/teams/GSW/2025.html#all_totals_stats (Update if needed)

In [ ]:
import pandas as pd
import re

ENRICHED_CSV = "enriched_data/nbastatsv3_2024_enriched_shots.csv"
GSW_ID = 1610612744  # Golden State Warriors

def load_enriched_clean(path: str) -> pd.DataFrame:
    # 1) Read as strings to avoid parse errors
    df = pd.read_csv(
        path,
        dtype=str,
        low_memory=False,
        on_bad_lines="skip",      # remove if your pandas is older
        encoding_errors="ignore"
    )

    # 2) Strip whitespace across object columns
    df = df.apply(lambda s: s.str.strip() if s.dtype == "object" else s)

    # 3) Drop repeated header rows appearing mid-file
    key_cols = [c for c in ["teamId", "gameId", "shotResult", "shotValue", "isFieldGoal", "actionId"] if c in df.columns]
    if key_cols:
        mask = pd.Series(False, index=df.index)
        for c in key_cols:
            mask = mask | df[c].eq(c)
        df = df[~mask]

    # 4) Normalize casing and types
    if "shotResult" in df.columns:
        df["shotResult"] = df["shotResult"].str.upper()

    for col in ["teamId", "isFieldGoal", "shotValue"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 5) **Normalize gameId** to NBA 10-digit format (e.g., 0022000001)
    #    - extract digits
    #    - zero-pad to length 10
    if "gameId" in df.columns:
        digits = df["gameId"].astype(str).str.extract(r"(\d+)", expand=False)
        df["gameId_norm"] = digits.fillna("").apply(lambda x: x.zfill(10) if x else x)
        # keep rows that have a normalized gameId
        df = df[df["gameId_norm"] != ""]
    else:
        df["gameId_norm"] = pd.NA  # fallback

    # 6) Keep rows with necessary fields
    df = df.dropna(subset=[c for c in ["teamId", "shotResult", "gameId_norm"] if c in df.columns])

    # 7) Deduplicate events (same game & action & team)
    if all(c in df.columns for c in ["gameId_norm", "actionId", "teamId"]):
        df = df.drop_duplicates(subset=["gameId_norm", "actionId", "teamId"])

    # 8) Restrict to **regular season** games only: gameId starts with "002"
    df = df[df["gameId_norm"].str.startswith("002")]

    # 9) Use pandas nullable integers
    for col in ["teamId", "isFieldGoal", "shotValue"]:
        if col in df.columns:
            df[col] = df[col].astype("Int64")

    return df

# --- Load & compute ---
season_enriched = load_enriched_clean(ENRICHED_CSV)
gsw = season_enriched[season_enriched["teamId"] == GSW_ID]

# Made field goals
gsw_fg_made = gsw[(gsw["isFieldGoal"] == 1) & (gsw["shotResult"] == "MADE")]

fgm = len(gsw_fg_made)
fg3m = (gsw_fg_made["shotValue"] == 3).sum()
fg2m = fgm - fg3m
pts = 2 * fg2m + 3 * fg3m

print("GSW regular-season field-goal totals (after clean + dedupe + ID normalization):")
print(f"PTS: {pts} | FGM: {fgm} | FG3M: {fg3m} | FG2M: {fg2m}")

# Diagnostics
prefix_counts = (season_enriched.assign(prefix=season_enriched["gameId_norm"].str[:3])
                 .groupby("prefix").size())
print("\nRows by normalized gameId prefix:")
print(prefix_counts)

print("\nSample normalized gameIds (first 10):")
print(season_enriched["gameId_norm"].drop_duplicates().head(10).to_list())

## 10. Sanity Check — Missing `shotValue`

In [ ]:

missing = season_shots[(season_shots["actionType"].str.contains("Shot", case=False, na=False)) &
                      (season_shots["isFieldGoal"]==1)]
# Join with enriched data if available
try:
    enriched = pd.read_csv(OUTPUT_CSV)
    missing_sv = enriched[(enriched["isFieldGoal"]==1) & (enriched["shotValue"].isna())]
    print("Total FGs:", len(enriched[enriched["isFieldGoal"]==1]))
    print("FGs missing shotValue:", len(missing_sv))
    display(missing_sv.head())
except Exception as e:
    print("Could not run sanity check:", e)

## 11. Sanity Check - GSW 3PA and 2PA

In [ ]:
# Load the CSV file
df = pd.read_csv("enriched_data/nbastatsv3_2024_enriched_shots.csv")
df=df[df["teamId"] == 1610612744]

# gsw = season_enriched[season_enriched["teamId"] == GSW_ID]

# Get unique values with counts
shotValue_counts = df["shotValue"].value_counts().reset_index()
shotValue_counts.columns = ["shotValue", "count"]

shotValue_counts